# Un extrait contient voix et percussion : changer de cible

Passer d’une classe exclusive à des présences indépendantes avec BCEWithLogitsLoss.

Nous lisons et exécutons les étapes dans l’ordre. Les cellules « Observer » montrent les valeurs et les formes ; elles ne prouvent pas la réussite des exercices.

Environnement vérifié : Python 3.11.9, PyTorch 2.10.0 CPU et TorchAudio 2.10.0 lorsque le chapitre le demande. Aucun modèle ni jeu de données n’est téléchargé pendant l’exécution.

[Revenir au chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#pertes-multi-etiquettes).


## Préparer le noyau

Choisissez un environnement CPU neuf. En cas de bibliothèque manquante, installez-la dans ce noyau, puis redémarrez-le. Les sorties affichent les versions réellement utilisées.

In [1]:
import os
import sys
import tempfile

# Choisir un noyau neuf pour garantir la vérification sur CPU.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
try:
    import torch
except ImportError as error:
    raise RuntimeError("PyTorch manque. Installer torch==2.10.0 dans ce noyau, puis redémarrer et relancer.") from error
torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", sys.version.split()[0], "— PyTorch", torch.__version__, "— CPU")
assert not torch.cuda.is_available(), "Redémarrer dans un noyau CPU neuf."
_quiz_initial_directory = os.getcwd()
_quiz_temporary_directory = tempfile.TemporaryDirectory(prefix="quiz-progressif-")
os.chdir(_quiz_temporary_directory.name)


Python 3.11.9 — PyTorch 2.10.0+cpu — CPU


In [2]:
# Cet outil sert uniquement à afficher les états des démonstrations.
# Il ne modifie pas les tenseurs et ne fait pas partie du modèle étudié.
def observer(nom, valeur):
    if isinstance(valeur, torch.Tensor):
        contenu = valeur.detach().cpu()
        print(nom, "forme", tuple(contenu.shape), "type", contenu.dtype,
              "requires_grad", valeur.requires_grad)
        if contenu.numel() <= 16:
            print("  valeurs :", contenu.tolist())
    elif valeur is None or isinstance(valeur, (int, float, bool, str, tuple)):
        print(nom, "=", valeur)
    elif isinstance(valeur, list) and len(valeur) <= 8:
        if all(isinstance(item,(int,float,bool,str)) for item in valeur):
            print(nom, "=", valeur)
        else:
            print(nom, ":",len(valeur),"éléments ; formes des tenseurs :",
                  [tuple(item.shape) for item in valeur if isinstance(item,torch.Tensor)])
    elif isinstance(valeur,dict):
        print(nom, ": clés",list(valeur)[:12])

def observer_entrainement(espace, montrer_poids=False):
    # Une copie évite de parcourir un dictionnaire global en mutation.
    vus = set()
    for nom, objet in list(espace.items()):
        if nom.startswith('_'):
            continue
        if isinstance(objet, torch.Tensor) and objet.is_leaf and objet.requires_grad:
            parametres = [(nom, objet)]
        elif isinstance(objet, torch.nn.Module):
            parametres = [(nom + '.' + cle, parametre)
                          for cle, parametre in list(objet.named_parameters())[:4]]
        else:
            continue
        for cle, parametre in parametres:
            if id(parametre) in vus:
                continue
            vus.add(id(parametre))
            if montrer_poids:
                observer(cle, parametre)
            observer(cle + '.grad', parametre.grad)


Dans le socle, chaque extrait avait une classe exclusive. Une percussion superposée à une voix pose un autre problème : nous voulons détecter les deux présences. Cette fiche devient utile lorsque votre annotation change ainsi.

| Tâche | Cible pour un extrait | Sens |
| --- | --- | --- |
| Exclusive | [1], entier long | Cet extrait est de classe percussion. |
| Présences | [[1.,1.]], flottants | Cet extrait contient voix et percussion. |

Les cibles multi-étiquettes ont la même forme que les scores : ici (1,2). Chaque colonne répond oui ou non indépendamment.

Choisissons deux scores nuls pour partir d’une indécision simple. La sigmoïde vaut alors 0,5 pour chaque colonne. Ces probabilités ne se partagent pas une somme de un : elles décrivent deux questions différentes.

Le critère reçoit directement les logits. Il intègre le calcul binaire associé à la sigmoïde de manière stable ; ajouter une sigmoïde avant cet appel changerait le calcul attendu. reduction="none" garde les deux coûts visibles.

Chaque présence vraie coûte ici −ln(0,5), soit environ 0,69. Le logarithme est naturel. La moyenne vaut donc aussi 0,69. Les deux gradients valent −0,25 : diminuer la perte pousserait les deux scores vers le haut.

Le seuil 0,5 est choisi pour cette démonstration ; avec ≥, une égalité est retenue. Il donne ici deux décisions vraies. Un argmax n’en conserverait qu’une. En projet, le seuil se choisit sur la validation et peut varier par étiquette.

## Démonstration 1 · Suivre deux présences indépendantes

Comparer le coût, les décisions et les gradients d’un même mélange.

### Préparer les bibliothèques

Exécutez cette petite partie après la précédente. Si un objet est créé, la cellule d’observation suivante permet d’en examiner l’état.

In [3]:
import torch
from torch import nn


### Calculer l’état suivant

Une ligne contient deux présences vraies ; les cibles sont flottantes.

In [4]:
scores = torch.tensor([[0., 0.]], requires_grad=True)
presences = torch.tensor([[1., 1.]], dtype=torch.float32)


In [5]:
observer('scores', scores)
observer('presences', presences)

scores forme (1, 2) type torch.float32 requires_grad True
  valeurs : [[0.0, 0.0]]
presences forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[1.0, 1.0]]


### Calculer l’état suivant

Le critère reçoit les scores bruts ; mean rassemble les deux contributions.

In [6]:
critere = nn.BCEWithLogitsLoss(reduction="none")
par_etiquette = critere(scores, presences)
perte = par_etiquette.mean()


In [7]:
observer('par_etiquette', par_etiquette)
observer('perte', perte)

par_etiquette forme (1, 2) type torch.float32 requires_grad True
  valeurs : [[0.6931471824645996, 0.6931471824645996]]
perte forme () type torch.float32 requires_grad True
  valeurs : 0.6931471824645996


### Calculer l’état suivant

Le seuil s’applique indépendamment, avec égalité acceptée ici.

In [8]:
probabilites = scores.sigmoid()
decisions = probabilites >= 0.5


In [9]:
observer('probabilites', probabilites)
observer('decisions', decisions)

probabilites forme (1, 2) type torch.float32 requires_grad True
  valeurs : [[0.5, 0.5]]
decisions forme (1, 2) type torch.bool requires_grad False
  valeurs : [[True, True]]


### Calculer les gradients

Les deux présences attendues poussent les scores vers le haut.

In [10]:
perte.backward()


In [11]:
observer_entrainement({'scores': scores, 'presences': presences, 'critere': critere, 'par_etiquette': par_etiquette, 'perte': perte, 'probabilites': probabilites, 'decisions': decisions}, montrer_poids=False)

scores.grad forme (1, 2) type torch.float32 requires_grad False
  valeurs : [[-0.25, -0.25]]


### Lire les sorties

Ces impressions lisent les objets calculés. Elles n’entraînent pas le modèle et ne modifient pas les tenseurs.

In [12]:
print('probabilites :', probabilites.tolist())
print('cout par presence :', par_etiquette.detach().tolist())
print('decisions :', decisions.tolist())
print('gradient des scores :', scores.grad.tolist())

probabilites : [[0.5, 0.5]]
cout par presence : [[0.6931471824645996, 0.6931471824645996]]
decisions : [[True, True]]
gradient des scores : [[-0.25, -0.25]]


In [13]:
assert bool(par_etiquette.shape == scores.shape == presences.shape == (1, 2)), 'pertes-multi-etiquettes/main: par_etiquette.shape == scores.shape == presences.shape == (1, 2)'
assert bool(torch.allclose(par_etiquette, torch.log(torch.full((1, 2), 2.)))), 'pertes-multi-etiquettes/main: torch.allclose(par_etiquette, torch.log(torch.full((1, 2), 2.)))'
assert bool(torch.allclose(scores.grad, torch.tensor([[-0.25, -0.25]]))), 'pertes-multi-etiquettes/main: torch.allclose(scores.grad, torch.tensor([[-0.25, -0.25]]))'
assert bool(decisions.tolist() == [[True, True]]), 'pertes-multi-etiquettes/main: decisions.tolist() == [[True, True]]'
print("Contrôles numériques de la démonstration réussis.")

Contrôles numériques de la démonstration réussis.


Le contrat de la perte découle du sens de l’annotation.

## À vous de modifier le calcul

Les deux cellules suivantes sont prêtes à éditer. Si vous les exécutez sans les compléter, elles le signalent et ne prétendent pas que vous avez réussi. Une réponse fausse déclenche le contrôle correspondant.

### Exercice 1

Un seul extrait contient voix ET percussion. Écrivez sa cible multi-étiquettes flottante dans cet ordre.

In [14]:
exo_logits = torch.tensor([[.2,-.1]])
exo_cible = ...  # Remplacer les points par votre réponse.
if exo_cible is Ellipsis or exo_cible is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.as_tensor(exo_cible).dtype.is_floating_point
    assert torch.equal(torch.as_tensor(exo_cible),torch.ones_like(exo_logits))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 1</summary>

Deux présences simultanées donnent deux 1, dans une ligne de forme (1,2).

</details>

### Exercice 2

Ces deux probabilités sigmoïdes doivent-elles nécessairement sommer à un ? Répondez par True ou False.

In [15]:
exo_probabilites = torch.sigmoid(torch.tensor([[2.,2.]]))
exo_somme_un = ...  # Remplacer les points par votre réponse.
if exo_somme_un is Ellipsis or exo_somme_un is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_somme_un is bool(torch.isclose(exo_probabilites.sum(),torch.tensor(1.)).item())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")


À compléter : cet exercice n’est pas encore réalisé.


<details>
<summary>Indice pour l’exercice 2</summary>

Chaque présence est traitée séparément. La softmax répond à une autre convention.

</details>

## Comparer avec les solutions

Ouvrez-les après avoir formulé votre réponse et votre explication. Les solutions sont également exécutées par le vérificateur du dépôt.

<details>
<summary>Solution de l’exercice 1</summary>

```python
exo_logits = torch.tensor([[.2,-.1]])
exo_cible = torch.tensor([[1.,1.]])
if exo_cible is Ellipsis or exo_cible is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert torch.as_tensor(exo_cible).dtype.is_floating_point
    assert torch.equal(torch.as_tensor(exo_cible),torch.ones_like(exo_logits))
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

<details>
<summary>Solution de l’exercice 2</summary>

```python
exo_probabilites = torch.sigmoid(torch.tensor([[2.,2.]]))
exo_somme_un = False
if exo_somme_un is Ellipsis or exo_somme_un is None:
    print("À compléter : cet exercice n’est pas encore réalisé.")
else:
    assert exo_somme_un is bool(torch.isclose(exo_probabilites.sum(),torch.tensor(1.)).item())
    print("Contrôle réussi pour la réponse fournie. Expliquez maintenant pourquoi.")
```

</details>

In [18]:
os.chdir(_quiz_initial_directory)
_quiz_temporary_directory.cleanup()